In [662]:
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
# q.grad.zero_()

In [663]:
######################## explanation
## "The example model is based on a 5-nodes grid shell strcuture
##  which has a crossing projection in the x-y plane" 
##  The FE part would be based on the Euler-bernoulli beam theory

In [664]:
## Initial Geom
# Initial Coords:
node_coords = torch.tensor([[0.0, 3.0, 0.0],[3.0, 0.0, 0.0],[3.0, 3.0, 0.0],[3.0, 6.0, 0.0],[9.0, 0.0, 0.0],[9.0, 3.0, 0.0],[9.0, 6.0, 0.0],[12.0, 3.0, 0.0]],
                               dtype=torch.float32, requires_grad=True,)
connectivity = [[1,3],[2,3],[4,3],[3,6],[5,6],[7,6],[8,6]]
n_dof_per_node = 6  # Degrees of freedom per node
total_dof = n_dof_per_node * len(node_coords)
Fixed_nodes = [1,2,4,5,7,8]
Free_nodes = []
n_elements = len(connectivity)
n_nodes = len(node_coords)
for i in range(1, n_nodes + 1):
    if i not in Fixed_nodes:
        Free_nodes.append(i) 
idx_CF = [node - 1 for node in Fixed_nodes]
idx_CN = [i for i in range(n_nodes) if i not in idx_CF]

In [665]:
## Force condition
####### Force condition -> need for modification
## +0 to x direction ; +1 to y direction ; +2 to Z direction
## +3 Bending around the z axis ; +4 bending around y axis ; +5 Twisting
# implement upwards loading of 50 kN at every free nodes
F = torch.zeros(total_dof, dtype=torch.float32)
f_n = Free_nodes
f_type = [2,2] # The force type
F_value = torch.tensor([50.0, 50.0])*1000  # The force value/direction
r = 1 / torch.max(F_value)
print(r)
F_sign = torch.sign(F_value)
for idx, i in enumerate(f_n):
    F[6 * (i - 1) + f_type[idx]] = F_value[idx]  # unit: KN / KN*m

px = torch.zeros(len(f_n), 1, dtype=torch.float32)
py = torch.zeros(len(f_n), 1, dtype=torch.float32)
pz = torch.zeros(len(f_n), 1, dtype=torch.float32)

for i in range(len(f_n)):
    if f_type[idx] == 0 :
        px[i] = F_value[i]
    elif f_type[idx] == 1 :
        py[i] = F_value[i]
    elif f_type[idx] == 2 :
        pz[i] = F_value[i]

tensor(2.0000e-05)


In [666]:
# Optimized Variables
## initial variables
q = torch.rand(n_elements, requires_grad=True) 
Q = torch.diag(q) * 1/r # scaling

In [667]:
## Obtaining Matrix D
# Assembly the nodal-branch matrix:
C = torch.zeros(n_elements, n_nodes, dtype=torch.float32)
for n, (i, j) in enumerate(connectivity):
    C[n, i-1] = 1
    C[n, j-1] = -1
CF = C[:, idx_CF]  
CN = C[:, idx_CN] 
# Assembly D
Dn = torch.matmul(torch.transpose(CN, 0, 1),torch.matmul(Q, CN))
DF = torch.matmul(torch.transpose(CN, 0, 1),torch.matmul(Q, CF))

In [668]:
## Coordinates solution
# separate the coordinates:
fixed_idces = [node - 1 for node in Fixed_nodes]
xF = node_coords[fixed_idces, 0].unsqueeze(1)
yF = node_coords[fixed_idces, 1].unsqueeze(1)
zF = node_coords[fixed_idces, 2].unsqueeze(1)
xN = torch.matmul(torch.inverse(Dn),(px - torch.matmul(DF,xF)))
yN = torch.matmul(torch.inverse(Dn),(py - torch.matmul(DF,yF)))
zN = torch.matmul(torch.inverse(Dn),(pz - torch.matmul(DF,zF)))

In [669]:
## Coordinates renewal:
new_node_coords = node_coords.clone()
for idx, node in enumerate(Free_nodes):
    node_idx = node - 1  
    new_node_coords[node_idx, 0] = xN[idx] 
    new_node_coords[node_idx, 1] = yN[idx]  
    new_node_coords[node_idx, 2] = zN[idx]
print(new_node_coords)

tensor([[ 0.0000,  3.0000,  0.0000],
        [ 3.0000,  0.0000,  0.0000],
        [ 3.6721,  3.6448,  0.5582],
        [ 3.0000,  6.0000,  0.0000],
        [ 9.0000,  0.0000,  0.0000],
        [ 8.6055,  3.1287,  0.5515],
        [ 9.0000,  6.0000,  0.0000],
        [12.0000,  3.0000,  0.0000]], grad_fn=<CopySlices>)


In [670]:
## Check point: FDM completed
# if q.grad is not None:
#     q.grad.zero_()
# loss = torch.sum(new_node_coords[:, 2])
# loss.backward()
# print("Gradient of q:")
# print(q.grad)

In [671]:
############ Now comes to FE part:

In [672]:
#### Default values (in meters, Pascals, etc.)
## For rectangular sections
# D_width = 0.1
# D_height = 0.2
# D_young_modulus = 210e9  # Young's modulus in Pascals
# D_shear_modulus = 81e9  # Shear modulus in Pascals
# D_poisson_ratio = 0.3
# cross_section_angle_a = 0  # Cross-section angle at node 1 (anti-clockwise)
# cross_section_angle_b = 0  # Cross-section angle at node 2 (anti-clockwise)
# a_small_number = 1e-10

D_radius = 0.01
D_young_modulus = 210e9  # Young's modulus in Pascals
D_shear_modulus = 81e9  # Shear modulus in Pascals
D_poisson_ratio = 0.3
cross_section_angle_a = 0  # Cross-section angle at node 1 (anti-clockwise)
cross_section_angle_b = 0  # Cross-section angle at node 2 (anti-clockwise)
a_small_number = 1e-10

In [673]:
def rotation(v, k, theta):
    """Rotation of vector v around axis k by angle theta."""
    k = k / torch.norm(k)  # Normalize k
    cross_product = torch.cross(k, v)
    dot_product = torch.dot(k, v)

    # Ensure theta is a tensor
    theta = torch.tensor(theta, dtype=torch.float32) if not isinstance(theta, torch.Tensor) else theta

    v_rotated = v * torch.cos(theta) + cross_product * torch.sin(theta) + k * dot_product * (1 - torch.cos(theta))
    return v_rotated

In [674]:
###### Element Creating (euler beam)
# For rectangular sections
# class Beam:
#     def __init__(self, node_coordinates, width=D_width, height=D_height, young_modulus=D_young_modulus,
#                  shear_modulus=D_shear_modulus, poisson_ratio=D_poisson_ratio, Beta_a=cross_section_angle_a,
#                  Beta_b=cross_section_angle_b):
#         self.node_coordinates = node_coordinates  # (2, 3) tensor for node coordinates

#         # Material and geometry
#         self.width = width
#         self.height = height
#         self.young_modulus = young_modulus
#         self.shear_modulus = shear_modulus
#         self.poisson_ratio = poisson_ratio

#         # Cross-sectional properties
#         self.length = torch.norm(self.node_coordinates[1] - self.node_coordinates[0])  # Length of the beam
#         self.Iy = (self.width * self.height ** 3) / 12
#         self.Iz = (self.width ** 3 * self.height) / 12
#         self.A = self.width * self.height
#         self.J = (self.width * self.height ** 3) / 3

#         # Stiffness components
#         self.S_u = self.young_modulus * self.A / self.length
#         self.S_v1a = 12 * self.young_modulus * self.Iy / (self.length ** 3)
#         self.S_v1b = 6 * self.young_modulus * self.Iy / (self.length ** 2)
#         self.S_v2a = 12 * self.young_modulus * self.Iz / (self.length ** 3)
#         self.S_v2b = 6 * self.young_modulus * self.Iz / (self.length ** 2)
#         self.S_theta1a = 6 * self.young_modulus * self.Iy / (self.length ** 2)
#         self.S_theta1b = 4 * self.young_modulus * self.Iy / self.length
#         self.S_theta1c = 2 * self.young_modulus * self.Iy / self.length
#         self.S_theta2a = 6 * self.young_modulus * self.Iz / (self.length ** 2)
#         self.S_theta2b = 4 * self.young_modulus * self.Iz / self.length
#         self.S_theta2c = 2 * self.young_modulus * self.Iz / self.length
#         self.S_Tr = self.shear_modulus * self.J / self.length

#         # Section rotations at the two ends
#         self.Beta_a = Beta_a
#         self.Beta_b = Beta_b

In [675]:
# For circular sections
class Beam:
    def __init__(self, node_coordinates, R=D_radius, young_modulus=D_young_modulus,
                 shear_modulus=D_shear_modulus, poisson_ratio=D_poisson_ratio, Beta_a=cross_section_angle_a,
                 Beta_b=cross_section_angle_b):
        self.node_coordinates = node_coordinates  # (2, 3) tensor for node coordinates

        # Material and geometry
        self.radius = R
        self.young_modulus = young_modulus
        self.shear_modulus = shear_modulus
        self.poisson_ratio = poisson_ratio

        # Cross-sectional properties
        self.length = torch.norm(self.node_coordinates[1] - self.node_coordinates[0])  # Length of the beam
        self.Iy = (torch.pi * self.radius ** 4) / 4
        self.Iz = self.Iy 
        self.A = torch.pi * self.radius ** 2
        self.J = (torch.pi * self.radius ** 4) / 2

        # Stiffness components
        self.S_u = self.young_modulus * self.A / self.length
        self.S_v1a = 12 * self.young_modulus * self.Iy / (self.length ** 3)
        self.S_v1b = 6 * self.young_modulus * self.Iy / (self.length ** 2)
        self.S_v2a = 12 * self.young_modulus * self.Iz / (self.length ** 3)
        self.S_v2b = 6 * self.young_modulus * self.Iz / (self.length ** 2)
        self.S_theta1a = 6 * self.young_modulus * self.Iy / (self.length ** 2)
        self.S_theta1b = 4 * self.young_modulus * self.Iy / self.length
        self.S_theta1c = 2 * self.young_modulus * self.Iy / self.length
        self.S_theta2a = 6 * self.young_modulus * self.Iz / (self.length ** 2)
        self.S_theta2b = 4 * self.young_modulus * self.Iz / self.length
        self.S_theta2c = 2 * self.young_modulus * self.Iz / self.length
        self.S_Tr = self.shear_modulus * self.J / self.length

        # Section rotations at the two ends
        self.Beta_a = Beta_a
        self.Beta_b = Beta_b
        
        
    def get_element_stiffness_matrix(self):
        """Element stiffness matrix."""
        K_element = torch.tensor([
            [self.S_u, 0, 0, 0, 0, 0, -self.S_u, 0, 0, 0, 0, 0],
            [0, self.S_v1a, 0, 0, 0, self.S_theta1a, 0, -self.S_v1a, 0, 0, 0, self.S_theta1a],
            [0, 0, self.S_v2a, 0, -self.S_theta2a, 0, 0, 0, -self.S_v2a, 0, -self.S_theta2a, 0],
            [0, 0, 0, self.S_Tr, 0, 0, 0, 0, 0, -self.S_Tr, 0, 0],
            [0, 0, -self.S_v2b, 0, self.S_theta2b, 0, 0, 0, self.S_v2b, 0, self.S_theta2c, 0],
            [0, self.S_v1b, 0, 0, 0, self.S_theta1b, 0, -self.S_v1b, 0, 0, 0, self.S_theta1c],
            [-self.S_u, 0, 0, 0, 0, 0, self.S_u, 0, 0, 0, 0, 0],
            [0, -self.S_v1a, 0, 0, 0, -self.S_theta1a, 0, self.S_v1a, 0, 0, 0, -self.S_theta1a],
            [0, 0, -self.S_v2a, 0, self.S_theta2a, 0, 0, 0, self.S_v2a, 0, self.S_theta2a, 0],
            [0, 0, 0, -self.S_Tr, 0, 0, 0, 0, 0, self.S_Tr, 0, 0],
            [0, 0, -self.S_v2b, 0, self.S_theta2c, 0, 0, 0, self.S_v2b, 0, self.S_theta2b, 0],
            [0, self.S_v1b, 0, 0, 0, self.S_theta1c, 0, -self.S_v1b, 0, 0, 0, self.S_theta1b],
        ], dtype=torch.float32)

        return K_element


    def System_Transform(self):
        """Coordinate transformation matrix."""
        vector_x = self.node_coordinates[1, 0] - self.node_coordinates[0, 0]
        vector_y = self.node_coordinates[1, 1] - self.node_coordinates[0, 1]
        vector_z = self.node_coordinates[1, 2] - self.node_coordinates[0, 2]
        length = torch.norm(self.node_coordinates[1] - self.node_coordinates[0])
        # print(vector_x,vector_y,vector_z)
        # Calculate alpha and ceta using PyTorch
        z_value = torch.clamp(vector_z / length, min=-1 + 1e-6, max=1 - 1e-6)
        ceta = torch.acos(z_value)
        value = vector_x / torch.sqrt(vector_y ** 2 + vector_x ** 2 + a_small_number)
        value = torch.clamp(value, min=-1 + 1e-6, max=1 - 1e-6)
        alpha = torch.acos(value)
        # print(alpha)

        Projection_Z_x = - vector_z / length * torch.sin(alpha)
        Projection_Z_y = - vector_z / length * torch.cos(alpha)
        Projection_Z_z = torch.cos(torch.pi / 2 - ceta)

        V_projection = torch.stack([Projection_Z_x, Projection_Z_y, Projection_Z_z])
        X_axis = torch.stack([vector_x / length, vector_y / length, vector_z / length])
        Z_axis_a = rotation(V_projection, X_axis, self.Beta_a)

        # Z_axis_b = rotation(V_projection, X_axis, self.Beta_b)
        Y_axis_a = rotation(Z_axis_a, X_axis, -torch.pi / 2)
        # Y_axis_b = rotation(Z_axis_b, X_axis, torch.pi / 2)
        Z_axis_a = Z_axis_a / torch.norm(Z_axis_a)
        # Z_axis_b = Z_axis_b / torch.norm(Z_axis_b)
        Y_axis_a = Y_axis_a / torch.norm(Y_axis_a)
        # Y_axis_b = Y_axis_b / torch.norm(Y_axis_b)

        lambda_matrix = torch.stack([X_axis, Y_axis_a, Z_axis_a], dim=0)
        matrix_T = torch.zeros((12, 12), dtype=torch.float32)
        for i in range(0, 12, 3):
            matrix_T[i:i + 3, i:i + 3] = lambda_matrix
        return matrix_T

    def nodal_transform(self):
        """Coordinate transformation matrix."""
        vector_x = self.node_coordinates[1, 0] - self.node_coordinates[0, 0]
        vector_y = self.node_coordinates[1, 1] - self.node_coordinates[0, 1]
        vector_z = self.node_coordinates[1, 2] - self.node_coordinates[0, 2]
        length = torch.norm(self.node_coordinates[1] - self.node_coordinates[0])
        # Calculate alpha and ceta using PyTorch
        z_value = torch.clamp(vector_z / length, min=-1 + 1e-6, max=1 - 1e-6)
        ceta = torch.acos(z_value)
        value = vector_x / torch.sqrt(vector_y ** 2 + vector_x ** 2 + a_small_number)
        value = torch.clamp(value, min=-1 + 1e-6, max=1 - 1e-6)
        alpha = torch.acos(value)

        Projection_Z_x = - vector_z / length * torch.sin(alpha)
        Projection_Z_y = - vector_z / length * torch.cos(alpha)
        Projection_Z_z = torch.cos(torch.pi / 2 - ceta)

        V_projection = torch.stack([Projection_Z_x, Projection_Z_y, Projection_Z_z])
        X_axis = torch.stack([vector_x / length, vector_y / length, vector_z / length])
        Z_axis_a = rotation(V_projection, X_axis, self.Beta_a)

        # Z_axis_b = rotation(V_projection, X_axis, self.Beta_b)
        Y_axis_a = rotation(Z_axis_a, X_axis, -torch.pi / 2)
        # Y_axis_b = rotation(Z_axis_b, X_axis, torch.pi / 2)
        Z_axis_a = Z_axis_a / torch.norm(Z_axis_a)
        # Z_axis_b = Z_axis_b / torch.norm(Z_axis_b)
        Y_axis_a = Y_axis_a / torch.norm(Y_axis_a)
        # Y_axis_b = Y_axis_b / torch.norm(Y_axis_b)

        lambda_matrix = torch.stack([X_axis, Y_axis_a, Z_axis_a], dim=0)

        return lambda_matrix



In [676]:
def assemble_stiffness_matrix(beams, n_nodes, n_dof_per_node, connectivity):
    """Global stiffness matrix assembly."""
    total_dof = n_nodes * n_dof_per_node  # Total degrees of freedom
    K_global = torch.zeros((total_dof, total_dof), dtype=torch.float32)
    for idx, (i, j) in enumerate(connectivity):
        Matrix_T = beams[idx].System_Transform()  # Get transformation matrix

        K_element = torch.matmul(torch.transpose(Matrix_T, 0, 1),
                                 torch.matmul(beams[idx].get_element_stiffness_matrix(), Matrix_T))

        start_idx = (i-1) * n_dof_per_node
        end_idx = (j-1) * n_dof_per_node
        K_global[start_idx:start_idx + 6, start_idx:start_idx + 6] = K_global[start_idx:start_idx + 6, start_idx:start_idx + 6] + K_element[0:6, 0:6]
        K_global[end_idx:end_idx + 6, end_idx:end_idx + 6] = K_global[end_idx:end_idx + 6,
                                                             end_idx:end_idx + 6] + K_element[6:12, 6:12]
        K_global[start_idx:start_idx + 6, end_idx:end_idx + 6] = K_global[start_idx:start_idx + 6,
                                                                 end_idx:end_idx + 6] + K_element[0:6, 6:12]
        K_global[end_idx:end_idx + 6, start_idx:start_idx + 6] = K_global[end_idx:end_idx + 6,
                                                                start_idx:start_idx + 6] + K_element[6:12, 0:6]


    return K_global

In [677]:
def Strain_E(node_coords, connectivity, fixed_dof, F):
    # node_coords[:, 2] = 0.0
    # Element Assembly
    beams = []
    for connection in connectivity:
        node_1_coords = node_coords[connection[0] - 1]
        node_2_coords = node_coords[connection[1] - 1]
        beam = Beam(node_coordinates=torch.stack([node_1_coords, node_2_coords]),
                    R=D_radius, young_modulus=D_young_modulus,
                    shear_modulus=D_shear_modulus, poisson_ratio=D_poisson_ratio, Beta_a=cross_section_angle_a,
                    Beta_b=cross_section_angle_b)
        beams.append(beam)
    # Stiffness renewal
    K_global = assemble_stiffness_matrix(beams, n_nodes=len(node_coords), n_dof_per_node=6, connectivity=connectivity)
    K_global[fixed_dof, :] = K_global[fixed_dof, :] - K_global[fixed_dof, :]
    K_global[:, fixed_dof] = K_global[:, fixed_dof] - K_global[:, fixed_dof]
    K_global[fixed_dof, fixed_dof] = 1e5
    rank_K = torch.linalg.matrix_rank(K_global)

    if rank_K < K_global.shape[0]:
        print("Warning: The stiffness matrix is not of full rank")
    else:
        displacements = torch.linalg.solve(K_global, F)

    # Compute strain energy
    strain_energy_list = []
    Local_d = torch.zeros(len(connectivity), 12, dtype=torch.float32)
    for n, (i, j) in enumerate(connectivity):
        matrix_T = beams[n].System_Transform()
        Tep_displacements = torch.cat([displacements[6 * (i-1):6 * (i-1) + 6], displacements[6 * (j-1):6 * (j-1) + 6]], dim=0)
        Local_d_n = torch.matmul(Tep_displacements, matrix_T.T)
        Local_d[n, :] = Local_d_n.clone()
        K_g = torch.matmul(torch.transpose(matrix_T, 0, 1),
                           torch.matmul(beams[n].get_element_stiffness_matrix(), matrix_T))

        strain_energy_list.append(torch.matmul(Local_d_n, torch.matmul(K_g, Local_d_n.reshape(-1, 1))))

    Strain_energy = torch.stack(strain_energy_list)

    return Strain_energy, Local_d, displacements, beams


In [678]:
# Element creating
#################  input below:
############################################################################################################## input
# initial coordinates
N_coords = new_node_coords.clone()
####### BCs
fixed_dof = []
for node in Fixed_nodes:
    fixed_dof.extend([(node-1) * 6 + i for i in range(6)])
# print(fixed_dof)
####### Gradient descent
step = 0.5
epochs = 50

####### Plotting
steps = 20  # steps for tracing deformation shape
###################################################################################################################################################
###################################################################################################################################################

In [679]:
## Optimization
torch.autograd.set_detect_anomaly(True)
# Optimizer setting
optimizer = optim.Adam([q], lr=step)
# Loop start
for iteration in range(epochs + 1):
    print(iteration)
    # Forwards
    Strain_energy, _, displacements, _ = Strain_E(N_coords, connectivity, fixed_dof, F)
#     print(displacements)
    Total_EStrain = torch.sum(Strain_energy)
#     print(Total_EStrain)
    # Backwards
    optimizer.zero_grad()
    print(q.grad)
    Total_EStrain.backward(retain_graph=True)

    # Grad
    gradients = q.grad
    frob_norm = torch.norm(gradients)
    print(gradients)
    optimizer.step()
    print(q)
    print(new_node_coords)
    # print iteration
    if iteration % 5 == 0:
        print(f"Iteration {iteration}: Normalized Gradient = {frob_norm}, Adaptive learning rate = {step}")
print("Optimization completed.")
# print(q)
# print("coordinates:",node_coords)

0
None
tensor([-1241.4803,  2929.1250,  1342.8379,  6192.6650,   444.2154,   982.7972,
          450.7437])
tensor([ 0.9947, -0.0935,  0.3836,  0.0440,  0.0902,  0.0746,  0.1552],
       requires_grad=True)
tensor([[ 0.0000,  3.0000,  0.0000],
        [ 3.0000,  0.0000,  0.0000],
        [ 3.6721,  3.6448,  0.5582],
        [ 3.0000,  6.0000,  0.0000],
        [ 9.0000,  0.0000,  0.0000],
        [ 8.6055,  3.1287,  0.5515],
        [ 9.0000,  6.0000,  0.0000],
        [12.0000,  3.0000,  0.0000]], grad_fn=<CopySlices>)
Iteration 0: Normalized Gradient = 7186.080078125, Adaptive learning rate = 0.5
1
None
tensor([-1241.4803,  2929.1250,  1342.8379,  6192.6650,   444.2154,   982.7972,
          450.7437])
tensor([ 1.4947, -0.5935, -0.1164, -0.4560, -0.4098, -0.4254, -0.3448],
       requires_grad=True)
tensor([[ 0.0000,  3.0000,  0.0000],
        [ 3.0000,  0.0000,  0.0000],
        [ 3.6721,  3.6448,  0.5582],
        [ 3.0000,  6.0000,  0.0000],
        [ 9.0000,  0.0000,  0.0000],
  

None
tensor([-1241.4803,  2929.1250,  1342.8379,  6192.6650,   444.2154,   982.7972,
          450.7437])
tensor([ 8.9947, -8.0935, -7.6164, -7.9560, -7.9098, -7.9254, -7.8448],
       requires_grad=True)
tensor([[ 0.0000,  3.0000,  0.0000],
        [ 3.0000,  0.0000,  0.0000],
        [ 3.6721,  3.6448,  0.5582],
        [ 3.0000,  6.0000,  0.0000],
        [ 9.0000,  0.0000,  0.0000],
        [ 8.6055,  3.1287,  0.5515],
        [ 9.0000,  6.0000,  0.0000],
        [12.0000,  3.0000,  0.0000]], grad_fn=<CopySlices>)
17
None
tensor([-1241.4803,  2929.1250,  1342.8379,  6192.6650,   444.2154,   982.7972,
          450.7437])
tensor([ 9.4947, -8.5935, -8.1164, -8.4560, -8.4098, -8.4254, -8.3448],
       requires_grad=True)
tensor([[ 0.0000,  3.0000,  0.0000],
        [ 3.0000,  0.0000,  0.0000],
        [ 3.6721,  3.6448,  0.5582],
        [ 3.0000,  6.0000,  0.0000],
        [ 9.0000,  0.0000,  0.0000],
        [ 8.6055,  3.1287,  0.5515],
        [ 9.0000,  6.0000,  0.0000],
        [

None
tensor([-1241.4803,  2929.1250,  1342.8379,  6192.6650,   444.2154,   982.7972,
          450.7437])
tensor([ 16.4947, -15.5935, -15.1164, -15.4560, -15.4098, -15.4254, -15.3448],
       requires_grad=True)
tensor([[ 0.0000,  3.0000,  0.0000],
        [ 3.0000,  0.0000,  0.0000],
        [ 3.6721,  3.6448,  0.5582],
        [ 3.0000,  6.0000,  0.0000],
        [ 9.0000,  0.0000,  0.0000],
        [ 8.6055,  3.1287,  0.5515],
        [ 9.0000,  6.0000,  0.0000],
        [12.0000,  3.0000,  0.0000]], grad_fn=<CopySlices>)
32
None
tensor([-1241.4803,  2929.1250,  1342.8379,  6192.6650,   444.2154,   982.7972,
          450.7437])
tensor([ 16.9947, -16.0935, -15.6164, -15.9560, -15.9098, -15.9254, -15.8448],
       requires_grad=True)
tensor([[ 0.0000,  3.0000,  0.0000],
        [ 3.0000,  0.0000,  0.0000],
        [ 3.6721,  3.6448,  0.5582],
        [ 3.0000,  6.0000,  0.0000],
        [ 9.0000,  0.0000,  0.0000],
        [ 8.6055,  3.1287,  0.5515],
        [ 9.0000,  6.0000,  0.00

None
tensor([-1241.4803,  2929.1250,  1342.8379,  6192.6650,   444.2154,   982.7972,
          450.7437])
tensor([ 23.9947, -23.0935, -22.6164, -22.9560, -22.9098, -22.9254, -22.8448],
       requires_grad=True)
tensor([[ 0.0000,  3.0000,  0.0000],
        [ 3.0000,  0.0000,  0.0000],
        [ 3.6721,  3.6448,  0.5582],
        [ 3.0000,  6.0000,  0.0000],
        [ 9.0000,  0.0000,  0.0000],
        [ 8.6055,  3.1287,  0.5515],
        [ 9.0000,  6.0000,  0.0000],
        [12.0000,  3.0000,  0.0000]], grad_fn=<CopySlices>)
47
None
tensor([-1241.4803,  2929.1250,  1342.8379,  6192.6650,   444.2154,   982.7972,
          450.7437])
tensor([ 24.4947, -23.5935, -23.1164, -23.4560, -23.4098, -23.4254, -23.3448],
       requires_grad=True)
tensor([[ 0.0000,  3.0000,  0.0000],
        [ 3.0000,  0.0000,  0.0000],
        [ 3.6721,  3.6448,  0.5582],
        [ 3.0000,  6.0000,  0.0000],
        [ 9.0000,  0.0000,  0.0000],
        [ 8.6055,  3.1287,  0.5515],
        [ 9.0000,  6.0000,  0.00